## Project IDM U1 group SE3 ##

### MCDA ###

Best-worst method (BWM) to determine weights 

In [9]:
import numpy as np
from scipy.optimize import linprog
import pandas as pd
from genetic_algorithm_pfm.a_fine_aggregator import a_fine_aggregator

In [10]:
def bwm_linear(a_B, a_W):
    n = len(a_B)
    B = np.argmin(a_B)
    W = np.argmin(a_W)
    
    c = np.zeros(n + 1)
    c[-1] = 1
    
    A_ub, b_ub = [], []
    for j in range(n):
        # w_B - a_Bj * w_j - xi <= 0
        row = np.zeros(n + 1); row[B] = 1; row[j] -= a_B[j]; row[-1] = -1
        A_ub.append(row); b_ub.append(0)
        # -w_B + a_Bj * w_j - xi <= 0
        row = np.zeros(n + 1); row[B] = -1; row[j] += a_B[j]; row[-1] = -1
        A_ub.append(row); b_ub.append(0)
        # w_j - a_jW * w_W - xi <= 0
        row = np.zeros(n + 1); row[j] = 1; row[W] -= a_W[j]; row[-1] = -1
        A_ub.append(row); b_ub.append(0)
        # -w_j + a_jW * w_W - xi <= 0
        row = np.zeros(n + 1); row[j] = -1; row[W] += a_W[j]; row[-1] = -1
        A_ub.append(row); b_ub.append(0)
    
    A_eq = [np.concatenate([np.ones(n), [0]])]
    b_eq = [1]
    bounds = [(0, None)] * n + [(0, None)]
    
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                  bounds=bounds, method='highs')
    return res.x[:n], res.x[-1]

# Voorbeeld RWS
a_B = [1, 4, 6, 5, 2, 4, 8, 3, 3, 3]
a_W = [8, 3, 2, 2, 6, 3, 1, 4, 3, 4]
w, xi = bwm_linear(a_B, a_W)
print("Gewichten:", np.round(w, 3))
print("xi*:", np.round(xi, 4))

Gewichten: [0.267 0.074 0.05  0.059 0.149 0.074 0.03  0.099 0.099 0.099]
xi*: 0.0297


In [11]:
df = pd.read_csv("Input_MCDA/MCDA_Ratings_Merwedebrug_final.csv", sep=";", skiprows = [1,2])

# display(df.head())
# df.keys()

criteria = df["Stakeholder"]
keys = df.keys()
for i in range(1, len(keys) - 1, 2):

    a_b_i = np.array(df.iloc[:, i])
    a_w_i = np.array(df.iloc[:, i + 1])
    a_b_i = list(a_b_i)
    a_w_i = list(a_w_i)

    to_pop = []

    for j in range(len(a_b_i)):
    
        if np.isnan(a_b_i[j]):
            to_pop.append(j)

    for pop in reversed(to_pop):
        a_b_i.pop(pop)  
        a_w_i.pop(pop)  

    a_b_i = np.array(a_b_i)
    a_w_i = np.array(a_w_i)
    
    w, xi = bwm_linear(a_b_i, a_w_i)
    print(f"Weights for criteria for stakeholder {keys[i]}", np.round(w, 3))
    print("xi*:", np.round(xi, 4))

Weights for criteria for stakeholder AB_RWS [0.267 0.074 0.05  0.059 0.149 0.074 0.03  0.099 0.099 0.099]
xi*: 0.0297
Weights for criteria for stakeholder AB_MIWM [0.12  0.323 0.12  0.09  0.18  0.072 0.06  0.036]
xi*: 0.0359
Weights for criteria for stakeholder AB_Municipality [0.104 0.078 0.104 0.264 0.157 0.157 0.104 0.031]
xi*: 0.0492
Weights for criteria for stakeholder AB_Road_users [0.184 0.123 0.325 0.156 0.057 0.156]
xi*: 0.0425
Weights for criteria for stakeholder AB_Residents [0.231 0.154 0.205 0.051 0.359]
xi*: 0.1026
Weights for criteria for stakeholder AB_Contractors [0.688 0.125 0.188]
xi*: 0.0625
Weights for criteria for stakeholder AB_Shipping_sector [0.204 0.204 0.102 0.102 0.041 0.347]
xi*: 0.0612


### Running the code ###

In [12]:
# Import packages
import numpy as np
import pandas as pd

# Round the float values in the dataframe to 2 decimal places
pd.options.display.float_format = '{:.2f}'.format

# Import local module for a-fine-aggregator
from genetic_algorithm_pfm.a_fine_aggregator import a_fine_aggregator

In [13]:
ratings = pd.read_csv("Input_MCDA/MCDA_alternatives.csv",
    sep=";",       # fields are semicolon-separated, not comma-separated
    skiprows=1     # skip the "MCDA Gold Coast Design Alternatives" title row
)

alternatives = list(ratings.columns[2:-1].unique()) # Get the list of alternatives from the dataframe columns, excluding the first two and last column
stakeholders = list(ratings["Stakeholder"].unique()) # Get the list of stakeholders from the "Stakeholder" column in the dataframe
print(f"Alternatives: {alternatives}")
print(f"Stakeholders: {stakeholders}")
display(ratings)

Alternatives: ['Bored Tunnel', 'Immersed tube tunnel', 'New Merwedebrug', 'Cable stay bridge']
Stakeholders: ['Rijkswaterstaat', 'MIWM', 'Municipalities', 'Road users', 'Residents', 'Contractor', 'Shipping sector']


,Stakeholder,Criteria,Bored Tunnel,Immersed tube tunnel,New Merwedebrug,Cable stay bridge,Criteria_Weight
0,Rijkswaterstaat,Operational safety,0,50,100,80,0.27
1,Rijkswaterstaat,Construction safety,0,30,100,70,0.07
2,Rijkswaterstaat,Life-cycle costs,0,20,100,50,0.05
3,Rijkswaterstaat,Construction time,30,0,100,80,0.06
4,Rijkswaterstaat,Reliability / service,60,100,80,0,0.15
5,Rijkswaterstaat,Accessibility,20,0,100,60,0.07
6,Rijkswaterstaat,Economic impact,70,100,0,40,0.03
7,Rijkswaterstaat,Capacity,0,40,100,60,0.10
8,Rijkswaterstaat,Environmental impact,70,100,0,50,0.10
9,Rijkswaterstaat,Navigability,100,90,40,0,0.10


In [14]:
# Check each stakeholder's weights sum to 1 (i.e. 100%)
print("Check weights per stakeholder:")
all_valid = True

for stakeholder in stakeholders:
    stakeholder_weights = ratings.loc[ratings["Stakeholder"] == stakeholder, "Criteria_Weight"]
    total = stakeholder_weights.sum()
    is_valid = np.isclose(total, 1)
    all_valid &= is_valid

    status = "OK" if is_valid else "MISMATCH"
    print(f"  {stakeholder:<25s}: {total:6.2f}  [{status}]")

Check weights per stakeholder:
  Rijkswaterstaat          :   1.00  [OK]
  MIWM                     :   1.00  [OK]
  Municipalities           :   1.00  [OK]
  Road users               :   1.00  [OK]
  Residents                :   1.00  [OK]
  Contractor               :   1.00  [OK]
  Shipping sector          :   1.00  [OK]


## Stakeholder Weights

Just as each stakeholder weighs their own criteria, the five stakeholder groups themselves must be weighted relative to one another before their individual preference scores can be combined into a single group score.

$$\sum_{i=1}^{5} w_i = 1$$

In this model, all stakeholder groups are initially weighted equally ($w_i = \tfrac{1}{5} = 0.2$), meaning no single stakeholder's preferences dominate the outcome. These weights can later be adjusted to reflect, for example, a decision-maker's judgement about whose interests should carry more weight.

In [15]:
# Set stakeholder weights
#               city, local, res, surf, tour
weights_eq =    [1/7, 1/7, 1/7, 1/7, 1/7, 1/7, 1/7]  # equal weights for stakeholders
weights_dom =   [10.3/40.9, 8.8/40.9, 5/40.9, 3.7/40.9, 3.2/40.9, 6.1/40.9, 3.8/40.9] # RWS dominant weight

stakeholder_weights = weights_eq

assert np.isclose(sum(stakeholder_weights), 1), f"Weights must sum to 1, got {sum(stakeholder_weights)}"


In [16]:
# Calculate the aggregated scores for each alternative using the a-fine-aggregator
# --- Level 1: aggregate criteria ratings -> one score per stakeholder per alternative ---
stakeholder_scores = {}

for stakeholder in stakeholders:
    stakeholder_data = ratings.loc[ratings["Stakeholder"] == stakeholder]
    criteria_weights = stakeholder_data["Criteria_Weight"].to_numpy()
    p = stakeholder_data[alternatives].to_numpy()  # shape: n_criteria x n_alternatives
    stakeholder_scores[stakeholder] = a_fine_aggregator(criteria_weights, p, scores_range=(-0.0, -100.0))

# Collect into matrix: rows = stakeholders, columns = alternatives (order matches `alternatives`)
stakeholder_score_matrix = np.array([stakeholder_scores[s] for s in stakeholders])

print("Individual stakeholder aggregated scores:")
display(pd.DataFrame(stakeholder_score_matrix, index=stakeholders, columns=alternatives))

# --- Level 2: aggregate stakeholder scores -> final preference score per alternative ---
final_scores = a_fine_aggregator(stakeholder_weights, stakeholder_score_matrix, scores_range=(-0.0, -100.0))

results = (
    pd.DataFrame(final_scores, index=alternatives, columns=["Preference score"])
    .round(2)
    .sort_values("Preference score", ascending=False)
)

print("Final aggregated preference scores per alternative:")
display(results)

Individual stakeholder aggregated scores:


,Bored Tunnel,Immersed tube tunnel,New Merwedebrug,Cable stay bridge
Rijkswaterstaat,0.00,56.20,100.00,41.76
MIWM,0.00,46.44,100.00,18.77
Municipalities,0.00,32.55,100.00,24.07
Road users,0.00,40.93,100.00,76.30
Residents,87.06,0.00,100.00,58.21
Contractor,0.00,100.00,92.95,58.05
Shipping sector,100.00,61.42,0.00,76.01


Final aggregated preference scores per alternative:


,Preference score
New Merwedebrug,100.00
Cable stay bridge,40.89
Immersed tube tunnel,37.02
Bored Tunnel,0.00
